# Getting Started

An introduction to `nuclearff`, the Nuclear Fantasy Football Python package. This notebook mirrors the [Getting Started](https://nolmacdonald.github.io/nuclearff/getting_started.html) docs page, with real cells run against a real Sleeper league.

**Running this notebook:** from the repo root, `uv sync --frozen --extra dev` once, then `uv run jupyter lab`. Every shell cell below uses `!uv run nuclearff ...` rather than a hardcoded venv path, so it always runs against this repo's current source and synced environment -- not a separately installed, possibly stale copy.

## Installation

`nuclearff` is not published to PyPI. From a clone of the repository:

```bash
git clone https://github.com/nolmacdonald/nuclearff.git
cd nuclearff
uv sync --frozen
```

`--frozen` installs exactly what `uv.lock` specifies. Install the dev extras to run linting, type checking, and tests (`uv sync --frozen --extra dev`), or the docs extras to build documentation (`uv sync --frozen --extra docs`).

## Configuration

Every command reads its settings from a configuration file, so a run can be reproduced from a git commit plus a config.

In [1]:
!uv run nuclearff config init -o ./configs/nuclearff.yaml --force

Wrote default configuration to configs/nuclearff.yaml


`paths.root` anchors every managed directory. This notebook passes `--root ./demo` on every command below, the same way the docs pages do -- it keeps this notebook's output self-contained under `examples/demo/`, off the real project data tree. (Don't reuse `./demo/data/raw` under a *fixed* `root/data/...` layout -- `paths.data` defaults to `data`, so `--root ./data` doubles into `./data/data/...`; a non-`data`-named root avoids that.)

In [2]:
!uv run nuclearff --root ./demo config paths --ensure

ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/raw
ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/processed
ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/cache
ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/manifests
ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts
ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/configs/leagues


## Capturing Your League

Fetch a complete, immutable snapshot of a real Sleeper league -- this project's own running example, `NUCLEARFF REDRAFT` (`1367225133634191360`).

In [3]:
!uv run nuclearff --root ./demo sleeper fetch-league --league-id 1367225133634191360

2026-09-09T23:17:09 [INFO] nuclearff.sleeper.snapshot: Wrote Sleeper snapshot to /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/raw/sleeper/1367225133634191360/20260910T061709Z
League:   NUCLEARFF REDRAFT (1367225133634191360)
Season:   2026 status=in_season
Teams:    10
Snapshot: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/raw/sleeper/1367225133634191360/20260910T061709Z
League config: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/configs/leagues/1367225133634191360.yaml

Settings to review:
  [WARNING] draft_rounds_mismatch: League settings report draft_rounds=3 but the draft object reports rounds=15.
           -> Trust the draft object. The league field is stale; use 15 rounds for pick-gap and VONA math.
  [WARNING] keepers_in_redraft: League type is 0 (redraft) but max_keepers=1.
           -> Treat the league as redraft, but confirm in the Sleeper UI whether a keeper is actually in play.
  [INFO] median_scoring: Each team also pla

The command writes each endpoint to `data/raw/sleeper/<league_id>/<timestamp>/` and never overwrites an existing capture. It also reports league settings that are contradictory or that change how players should be valued -- both warnings printed above are real for this league, not illustrative.

This is the starting point; `fetch-league` also has flags for multi-season history, standings and playoff results, weekly matchups, transaction history, and roster composition -- see `01_user_guide.ipynb` for a complete walkthrough.

## Data Sources

`nuclearff` builds on two data sources:

- **nflverse / nflreadpy** -- [nflreadpy](https://github.com/nflverse/nflreadpy) provides NFL play-by-play, player, roster, schedule, and advanced statistics data as `polars.DataFrame` objects, cached locally between calls.
- **Sleeper API** -- the [Sleeper API](https://docs.sleeper.com) is a read-only, unauthenticated HTTP API exposing leagues, rosters, matchups, transactions, and player metadata. See the [Sleeper API Tutorial](https://nolmacdonald.github.io/nuclearff/sleeper_api_tutorial.html) for a full walkthrough of fetching this data directly in Python.

## Quick Start

A minimal example independent of the CLI: weekly player statistics via `nflreadpy` directly.

In [4]:
import logging

import nflreadpy as nfl

from nuclearff import configure_logging

configure_logging(level=logging.INFO)

# Weekly player statistics for the 2023 and 2024 seasons
player_stats = nfl.load_player_stats([2023, 2024])
player_stats.head()

player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,…,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
str,str,str,str,str,str,i32,i32,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,str,str,str,i32,i32,i32,i32,i32,i32,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64
"""00-0023459""","""A.Rodgers""","""Aaron Rodgers""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2023,1,"""REG""","""2023_01_BUF_NYJ""","""NYJ""","""BUF""",0,1,0,0,0,1,-10,0,0,17,0,0,-2.03196,null,0,0.0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0023853""","""M.Prater""","""Matt Prater""","""K""","""SPEC""","""https://static.www.nfl.com/ima…",2023,1,"""REG""","""2023_01_ARI_WAS""","""ARI""","""WAS""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,"""28;54;37""",null,null,119,0,0,1,1,0,0,1.0,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0025565""","""N.Folk""","""Nick Folk""","""K""","""SPEC""","""https://static.www.nfl.com/ima…",2023,1,"""REG""","""2023_01_TEN_NO""","""TEN""","""NO""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,"""50;27;31;45;29""",null,null,182,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0026190""","""C.Campbell""","""Calais Campbell""","""DE""","""DL""","""https://static.www.nfl.com/ima…",2023,1,"""REG""","""2023_01_CAR_ATL""","""ATL""","""CAR""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0026498""","""M.Stafford""","""Matthew Stafford""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2023,1,"""REG""","""2023_01_LA_SEA""","""LA""","""SEA""",24,38,334,0,0,0,0,0,0,409,106,17,20.679982,-0.425926,0,0.816626,18,7,6,1,3,11,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,14.46,14.46


`nflreadpy` returns [polars](https://docs.pola.rs) DataFrames. Use `.to_pandas()` if a downstream tool expects a pandas DataFrame instead.

## Next steps

- `01_user_guide.ipynb` -- every other CLI command: league history, standings, matchups, transactions, roster composition, player filtering, auction/keeper valuation, and the draft board.
- `02_league_trade_history.ipynb` -- all 10 `report trades` visualizations.
- [User Guide](https://nolmacdonald.github.io/nuclearff/user_guide.html) and [League Trade History](https://nolmacdonald.github.io/nuclearff/league_trade_history.html) -- the matching docs pages.